# Labeling

See `labeling/GUIDE.md`. Enter your id, press Start, then click a class for each cell.

In [1]:
from pathlib import Path

import geopandas as gpd
import ipywidgets as W
from ipyleaflet import Map, Polygon, basemaps
from IPython.display import display

from hydroscope.data.eurosat import CLASSES
from hydroscope.data.labels import (
    SKIP,
    TARGET_PER_CLASS,
    class_counts,
    load_candidates,
    load_labels,
    resume_index,
    upsert_label,
)
from hydroscope.geo.grid import cell_outline_latlon

ROOT = Path.cwd() if (Path.cwd() / "labeling").is_dir() else Path.cwd().parent
AGS = ROOT / "data" / "ags"
LABELS_CSV = AGS / "labels.csv"
PREVIEWS = AGS / "previews"
# Read once, indexed by cell_id.
GRID = gpd.read_file(ROOT / "data" / "ref" / "grid.gpkg", layer="grid").set_index("cell_id")

In [2]:
state = {"i": 0, "annotator": "", "cands": None, "labels": None}

annotator_box = W.Text(description="Annotator", placeholder="your id")
start_btn = W.Button(description="Start", button_style="primary")
header = W.HTML()
counter = W.HTML()
image = W.Image(format="png", width=256, height=256)
# One map and one outline, updated per cell (a new Map per cell would pile up widgets).
fmap = Map(
    basemap=basemaps.Esri.WorldImagery, zoom=16, layout=W.Layout(width="400px", height="256px")
)
outline_layer = Polygon(locations=[], color="yellow", weight=2, fill=False)
fmap.add(outline_layer)
class_buttons = [W.Button(description=c) for c in CLASSES]
skip_btn = W.Button(description=SKIP, button_style="warning")
back_btn = W.Button(description="back")
done_html = W.HTML()


def refresh_counter():
    counts = class_counts(state["labels"])
    cells = []
    for c in CLASSES:
        n = counts[c]
        color = "#2e7d32" if n >= TARGET_PER_CLASS else "#555"
        style = f"color:{color};margin-right:12px"
        cells.append(f"<span style='{style}'>{c}: {n} / {TARGET_PER_CLASS}</span>")
    counter.value = "".join(cells)


def current_label():
    lab = state["labels"]
    cid = state["cands"].iloc[state["i"]]["cell_id"]
    hit = lab[(lab["cell_id"] == cid) & (lab["annotator"] == state["annotator"])]
    return hit.iloc[0]["label"] if len(hit) else None


def show():
    cands, i = state["cands"], state["i"]
    if i >= len(cands):
        header.value = f"<b>Done: {len(cands)} / {len(cands)}</b>"
        done_html.value = "All candidates answered."
        return
    done_html.value = ""
    row = cands.iloc[i]
    cid = row["cell_id"]
    cur = current_label()
    header.value = (
        f"<b>{i + 1} / {len(cands)}</b> &nbsp; {cid} &nbsp; block {row['block_id']}"
        f" &nbsp; current label: <b>{cur or '-'}</b>"
    )
    image.value = (PREVIEWS / f"{cid}.png").read_bytes()
    geom = GRID.loc[cid, "geometry"]
    outline = cell_outline_latlon(geom)
    lats = [p[0] for p in outline[:4]]
    lons = [p[1] for p in outline[:4]]
    center = (sum(lats) / 4, sum(lons) / 4)
    outline_layer.locations = outline
    fmap.center = center


def answer(label):
    if state["i"] >= len(state["cands"]):
        return
    row = state["cands"].iloc[state["i"]]
    state["labels"] = upsert_label(
        LABELS_CSV, row["cell_id"], label, state["annotator"], row["block_id"]
    )
    state["i"] += 1
    refresh_counter()
    show()


def go_back(_):
    if state["cands"] is not None and state["i"] > 0:
        state["i"] -= 1
        show()


def on_start(_):
    name = annotator_box.value.strip()
    if not name:
        header.value = "<b style='color:red'>Enter an annotator id first.</b>"
        return
    state["annotator"] = name
    state["cands"] = load_candidates(AGS / "candidates.csv", AGS / "cells.csv")
    state["labels"] = load_labels(LABELS_CSV)
    state["i"] = resume_index(state["cands"], state["labels"], name)
    for b in [*class_buttons, skip_btn, back_btn]:
        b.disabled = False
    refresh_counter()
    show()


for b in class_buttons:
    b.on_click(lambda _, label=b.description: answer(label))
skip_btn.on_click(lambda _: answer(SKIP))
back_btn.on_click(go_back)
start_btn.on_click(on_start)
for b in [*class_buttons, skip_btn, back_btn]:
    b.disabled = True  # enabled by Start; nothing is written before the first click

In [3]:
display(
    W.VBox(
        [
            W.HBox([annotator_box, start_btn]),
            header,
            W.HBox([image, fmap]),
            W.HBox(class_buttons[:5]),
            W.HBox(class_buttons[5:]),
            W.HBox([skip_btn, back_btn]),
            counter,
            done_html,
        ]
    )
)